In [3]:
import pandas as pd
import re
from cleaning import clean_phone_name

In [4]:
df1 = pd.read_csv('all_phones_final.csv')
df2 = pd.read_csv('cellphones_full.csv')

In [5]:
df2['Dung lượng RAM'].info()

<class 'pandas.Series'>
RangeIndex: 966 entries, 0 to 965
Series name: Dung lượng RAM
Non-Null Count  Dtype
--------------  -----
870 non-null    str  
dtypes: str(1)
memory usage: 7.7 KB


In [10]:
df1['Memory | Internal'].info()

<class 'pandas.Series'>
RangeIndex: 799 entries, 0 to 798
Series name: Memory | Internal
Non-Null Count  Dtype
--------------  -----
590 non-null    str  
dtypes: str(1)
memory usage: 6.4 KB


In [7]:
def add_ram(df_source, df_target, normalize_fn):
    def parse_ram_for_rom(mem_internal, rom_str):
        if pd.isna(mem_internal) or pd.isna(rom_str):
            return None
        rom_num = re.sub(r'[^0-9]', '', str(rom_str))
        pattern = rf'{rom_num}GB\s+(\d+)GB\s+RAM'
        m = re.search(pattern, str(mem_internal), re.IGNORECASE)
        if m:
            return f"{m.group(1)} GB"
        all_rams = re.findall(r'\d+GB\s+(\d+)GB\s+RAM', str(mem_internal), re.IGNORECASE)
        return f"{all_rams[0]} GB" if all_rams else None

    def find_match(norm_name, df_source):
        exact = df_source[df_source['_norm'] == norm_name]
        if len(exact) > 0:
            return exact.iloc[0]
        candidates = df_source[df_source['_norm'].str.contains(re.escape(norm_name), regex=True)]
        if len(candidates) > 0:
            return candidates.iloc[candidates['_norm'].str.len().argmin()]
        candidates2 = df_source[df_source['_norm'].apply(lambda x: x in norm_name and len(x) > 5)]
        if len(candidates2) > 0:
            return candidates2.iloc[candidates2['_norm'].str.len().argmax()]
        return None

    df_source = df_source.copy()
    df_target = df_target.copy()
    df_source['_norm'] = df_source['name_clean'].apply(normalize_fn)
    df_target['_norm'] = df_target['Tên'].apply(normalize_fn)

    for idx, row in df_target.iterrows():
        need_ram = pd.isna(row['Dung lượng RAM'])
        need_bat = pd.isna(row['Pin'])
        if not need_ram and not need_bat:
            continue

        match = find_match(row['_norm'], df_source)
        if match is None:
            continue

        if need_ram:
            ram = parse_ram_for_rom(match['Memory | Internal'], row['Bộ nhớ trong'])
            if ram:
                df_target.at[idx, 'Dung lượng RAM'] = ram

    return df_target.drop(columns=['_norm'])

In [8]:
df2 = add_ram(df1, df2, clean_phone_name)

In [12]:
df2['Dung lượng RAM'].info()

<class 'pandas.Series'>
RangeIndex: 966 entries, 0 to 965
Series name: Dung lượng RAM
Non-Null Count  Dtype
--------------  -----
891 non-null    str  
dtypes: str(1)
memory usage: 7.7 KB
